# Project 2

This project utilizes *openml* to read the dataset.

In [ ]:
import openml
import pandas as pd

# 1. Fetch the dataset using its ID
dataset = openml.datasets.get_dataset(31)

# 2. Get the data, specifying 'class' as the target variable
X, y, categorical_indicator, attribute_names = dataset.get_data(target="class")

df = pd.concat([X, y], axis=1)

## Data Preparation

The first step in perparing the data involes splitting the dataset into a test and training set.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

In [ ]:
# Display the first few rows
print(X.head())

# Check for data types and missing values
print(X.info())

# Statistical summary of numerical features
print(X.describe())

This output shows an overview of the German Credit dataset. This contains 1,000 entries across 20 different attributes. It includes a mix of categorical and numerical data types. 

#### Checking for missing values.

In [ ]:
# Check for null values
print(X.isnull().sum())

This shows there is no missing values in the dataset. 

#### Understanding the target variable.

In [ ]:
# Check class distribution
print(y.value_counts(normalize=True))

This shows us that the dataset is imbalanced with 70% having good credit, and 30% having bad credit.

### Visualizing the Dataset

To help understand the data, we use boxplots and histograms to get a better representation of the content in the dataset. This allows us to check for outliers and get more insight into the distribuion inside the dataset. First, looking at the histograms:

In [ ]:
import matplotlib.pyplot as plt

# Select numerical columns
num_cols = ['age', 'credit_amount', 'duration']

# Plot histograms
df[num_cols].hist(bins=20, figsize=(12, 8))
plt.suptitle('Distribution of Numerical Features')
plt.show()

The histograms illustrate that the **age** and **credit_amount** features exhibit a right-skewed distribution, showing that the majority of applicants are younger and request smaller loan amounts. Next, we examine the boxplots to check for outliers:

In [ ]:
import seaborn as sns

# Boxplot for credit_amount
plt.figure(figsize=(8, 5))
sns.boxplot(x=df['credit_amount'])
plt.title('Outlier Analysis: Credit Amount')
plt.show()

The boxplot reveals that the **credit_amount** feature is positively skewed, with a majority of loans concentrated at lower values.

Looking for correlations between attributes.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Encode categorical features temporarily for correlation analysis
df_encoded = df.copy()
for col in df_encoded.select_dtypes(['category', 'object']).columns:
    df_encoded[col] = df_encoded[col].cat.codes

# 2. Calculate the correlation matrix
corr_matrix = df_encoded.corr()

# 3. Plot the heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Attribute Correlation Heatmap')
plt.show()

### Preprocessing the Data

To prepare the dataset for machine learning we utilize a ColumnTransformer to standardize all the numerical features and apply one-hot encoding on all of the categorical features. This dual approach helps ensure that the features are correctly scaled and numerically represented for a consistent input for the model.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Identify feature types
numeric_features = ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']
categorical_features = ['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'job', 'own_telephone', 'foreign_worker']

# 2. Define the transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])

# 3. Fit and transform the data (using the training set)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

## Model Selection

Several classification algorithms are compared using cross-validation. Since the dataset is imbalanced, the weighted F1-score is used to select the best overall model.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import pandas as pd

# Candidate classification algorithms for credit risk prediction.
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
    "Support Vector Machine": SVC(class_weight="balanced", random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision_weighted", "recall_weighted", "f1_weighted"]

model_results = []
for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    model_results.append({
        "Model": model_name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision_weighted"].mean(),
        "Recall": scores["test_recall_weighted"].mean(),
        "F1 Score": scores["test_f1_weighted"].mean()
    })

results_df = pd.DataFrame(model_results).sort_values(by="F1 Score", ascending=False)
display(results_df)

best_model_name = results_df.iloc[0]["Model"]
print(f"Selected model: {best_model_name}")


## Model Training

The selected model is trained on the full training dataset so it can learn the relationship between applicant features and credit-risk category.

In [ ]:
best_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", models[best_model_name])
])

best_model.fit(X_train, y_train)

print(f"{best_model_name} has been trained on the full training dataset.")


## Model Evaluation

The trained model is evaluated on the unseen test set using accuracy, precision, recall, F1-score, and ROC-AUC.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from sklearn.preprocessing import LabelEncoder

# Predict credit-risk classes for the test data.
y_pred = best_model.predict(X_test)

# Get continuous scores for ROC-AUC.
classifier = best_model.named_steps["classifier"]
if hasattr(best_model, "predict_proba"):
    class_labels = list(classifier.classes_)
    positive_label = "bad" if "bad" in class_labels else class_labels[1]
    positive_index = class_labels.index(positive_label)
    y_scores = best_model.predict_proba(X_test)[:, positive_index]
elif hasattr(best_model, "decision_function"):
    class_labels = list(classifier.classes_)
    positive_label = "bad" if "bad" in class_labels else class_labels[1]
    y_scores = best_model.decision_function(X_test)
    if class_labels[1] != positive_label:
        y_scores = -y_scores
else:
    class_labels = sorted(y_test.unique())
    positive_label = "bad" if "bad" in class_labels else class_labels[1]
    y_scores = y_pred

# Convert target labels to binary values for ROC-AUC.
y_test_binary = (y_test == positive_label).astype(int)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
roc_auc = roc_auc_score(y_test_binary, y_scores)

evaluation_results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Score": [accuracy, precision, recall, f1, roc_auc]
})

display(evaluation_results)

print("Confusion Matrix:")
display(pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=[f"Actual {label}" for label in classifier.classes_],
    columns=[f"Predicted {label}" for label in classifier.classes_]
))

print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


## Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for Random Forest
param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [None, 5, 10, 15],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", "log2"]
}

# Build a fresh pipeline to tune (same structure as best_model)
tuning_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(class_weight="balanced", random_state=42))
])

grid_search = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best parameters found:")
print(grid_search.best_params_)
print(f"\nBest cross-validated F1 score: {grid_search.best_score_:.4f}")

# The tuned model
tuned_model = grid_search.best_estimator_

Hyperparameter tuning improved the cross-validated F1 score from 0.7532 (default Random Forest) to 0.7659, with the best configuration favoring a moderate tree depth (15), a larger ensemble (300 trees), and the log2 feature subset size

### Re-evaluating the tuned model

In [ ]:
# Predict with the tuned model
y_pred_tuned = tuned_model.predict(X_test)

tuned_classifier = tuned_model.named_steps["classifier"]
y_scores_tuned = tuned_model.predict_proba(X_test)[:, positive_index]

accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
precision_tuned = precision_score(y_test, y_pred_tuned, average="weighted", zero_division=0)
recall_tuned = recall_score(y_test, y_pred_tuned, average="weighted", zero_division=0)
f1_tuned = f1_score(y_test, y_pred_tuned, average="weighted", zero_division=0)
roc_auc_tuned = roc_auc_score(y_test_binary, y_scores_tuned)

comparison_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Before Tuning": [accuracy, precision, recall, f1, roc_auc],
    "After Tuning": [accuracy_tuned, precision_tuned, recall_tuned, f1_tuned, roc_auc_tuned]
})

display(comparison_df)

print("Confusion Matrix (Tuned Model):")
display(pd.DataFrame(
    confusion_matrix(y_test, y_pred_tuned),
    index=[f"Actual {label}" for label in tuned_classifier.classes_],
    columns=[f"Predicted {label}" for label in tuned_classifier.classes_]
))

print("Classification Report (Tuned Model):")
print(classification_report(y_test, y_pred_tuned, zero_division=0))

## Predictions

With the tuned Random Forest model finalized, we now generate credit-risk predictions for each applicant in the test set

In [ ]:
# 6. Prediction
# Use the tuned model to predict credit risk for the test applicants

# Predicted class labels
predictions = tuned_model.predict(X_test)

# Predicted probability of being "bad" credit risk
bad_probabilities = tuned_model.predict_proba(X_test)[:, positive_index]

# Build a results table combining key applicant info with predictions
results = X_test.copy()
results["Actual Credit Risk"] = y_test.values
results["Predicted Credit Risk"] = predictions
results["Predicted Probability (Bad)"] = bad_probabilities.round(3)

# Reset index for cleaner display
results = results.reset_index(drop=True)

print(f"Total applicants in test set: {len(results)}")
print(f"Predicted as 'bad' credit risk: {(results['Predicted Credit Risk'] == 'bad').sum()}")
print(f"Predicted as 'good' credit risk: {(results['Predicted Credit Risk'] == 'good').sum()}")

# Show a sample of predictions
results[["checking_status", "duration", "credit_amount", "age",
         "Actual Credit Risk", "Predicted Credit Risk", "Predicted Probability (Bad)"]].head(10)

Out of 200 test applicants, the model predicted 69 as "bad" credit risk and 131 as "good" slightly more than the 60 actual "bad" cases

In [ ]:
# Highlight the highest-risk applicants (highest predicted probability of "bad")
highest_risk = results.sort_values("Predicted Probability (Bad)", ascending=False).head(5)
print("Top 5 highest-risk applicants (by predicted probability):")
display(highest_risk[["checking_status", "duration", "credit_amount", "age",
                       "Actual Credit Risk", "Predicted Credit Risk", "Predicted Probability (Bad)"]])

Four of the five highest-risk predictions match the actual label, and all five applicants share a negative or low checking account balance, suggesting checking status is a strong driver of the model's risk assessment.

In [ ]:
# Cases where the model disagreed with the actual label — useful for later interpretation
misclassified = results[results["Actual Credit Risk"] != results["Predicted Credit Risk"]]
print(f"Number of misclassified applicants: {len(misclassified)} out of {len(results)}")
misclassified[["checking_status", "duration", "credit_amount", "age",
               "Actual Credit Risk", "Predicted Credit Risk", "Predicted Probability (Bad)"]].head(10)

The misclassifications include both false negatives, where applicants with a *"no checking"* status were predicted as low risk despite being actually bad, and false positives, where longer loan durations and larger credit amounts pushed otherwise good applicants into the *"bad"* risk category

## Interpertation